# 01. Named Entity & Topic Extraction — via LLM Prompting (Azure OpenAI Responses API)

This notebook demonstrates **entity and topic extraction using a general-purpose LLM prompt**, called through
an **Azure AI Foundry** project's OpenAI-compatible client (the Responses API), *not* the dedicated Azure AI
Language Named Entity Recognition (NER) service. The script sends a support-ticket-style piece of text to a
deployed chat model (`gpt-5.4`) with a system prompt instructing it to return entities and topics as JSON text.

This is a common real-world pattern: instead of (or alongside) calling a purpose-built NLP service, you ask an
LLM to do free-form extraction via prompt engineering. Compare this to `07_azure_language_ner.py`, which calls
the actual Azure AI Language `recognize_entities` prebuilt NER model — you'll see the two approaches produce
similar-looking output through very different mechanisms.

**Difficulty:** Beginner


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `openai` — the OpenAI Python SDK; the Foundry project hands you a pre-configured instance of this client
- `azure-ai-projects` — `AIProjectClient`, the entry point to an Azure AI Foundry project
- `azure-identity` — `DefaultAzureCredential`, Entra ID auth via your `az login` session (no API key needed)
- `python-dotenv` — loads `.env` into the process environment

No new installs needed for this notebook.

**Azure resources required:**
- An Azure AI Foundry project with a chat-capable model deployment (e.g. `gpt-5.4`, or any GPT-4o-class model
  you have deployed)

**Environment variables** (add these to the root `.env`, matching the pattern already used in `11_azure_ai_foundry/`):
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-deployment-name>
```

Auth is via `DefaultAzureCredential`, so you also need to be logged in locally: `az login`.


## What You'll Learn

- How to obtain an OpenAI-compatible client from an `AIProjectClient` (`get_openai_client()`)
- How to use a system prompt to instruct an LLM to perform entity/topic extraction
- The shape of the Azure OpenAI **Responses API** (`responses.create`) vs. the more familiar Chat Completions API
- Why prompt-based JSON extraction is *not* guaranteed to be valid, parseable JSON (motivating the structured
  output approach in notebook 02)
- How this LLM-prompting approach differs architecturally from calling the dedicated Azure AI Language NER
  service (notebook 07)


### Step 1 — Connect to the Azure AI Foundry project and get an OpenAI client

- `DefaultAzureCredential()` transparently tries several auth methods in order (environment vars, managed
  identity, Azure CLI login, etc.) — in local dev it typically falls back to your `az login` session.
- `AIProjectClient(endpoint, credential)` is the top-level handle to your Foundry **project** (as opposed to
  the raw Cognitive Services resource).
- `project.get_openai_client()` returns a standard `openai` SDK client object whose `base_url` and auth are
  already pointed at your Foundry project's model deployments — so everything downstream uses familiar
  OpenAI SDK syntax.

💡 **Exam tip:** For AI-102, know that Azure AI Foundry (formerly Azure AI Studio) projects can front multiple
model providers behind one client. The pattern of "get a client scoped to a project, then call it like the
underlying SDK" is central to how Foundry unifies access to OpenAI models, other model-catalog models, and
Foundry-hosted agents.

🔄 **Alternatives:** You could skip the Foundry project entirely and call `AzureOpenAI` directly from the
`openai` SDK with an Azure OpenAI resource endpoint + API key/Entra ID — useful if you don't need the broader
Foundry project features (agents, tracing, evaluation, connected data).


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT"]

project = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Step 2 — Define the sample text and the extraction prompt

The `ticket_text` stands in for a real support ticket. The `system_prompt` is doing all the "NLP work" here —
it tells the model exactly which entity categories to look for and the exact JSON shape to respond with.
This is **prompt engineering**, not a trained/certified extraction model — the model can still deviate from
the requested schema (which is exactly what notebook 02 fixes with a strict JSON schema).

💡 **Exam tip:** Contrast this ad hoc category list (`person`, `organization`, `date`, `product`, `ticket_id`,
`sla_tier`, `monetary_amount`) with Azure AI Language's prebuilt NER categories (`Person`, `Organization`,
`DateTime`, `Product`, `Quantity`, etc. — see notebook 07). Domain-specific categories like `ticket_id` or
`sla_tier` don't exist in the prebuilt model at all; getting them from Azure AI Language would require
**Custom Named Entity Recognition** (training your own model in Language Studio), whereas an LLM prompt can
invent arbitrary categories on the spot with zero training data.

🔄 **Alternatives:** Custom NER (train a small labeled dataset in Language Studio) if you need consistent,
auditable, low-latency extraction of domain-specific entities at scale, rather than paying LLM-call cost and
latency for every document.


In [ ]:
ticket_text = """
Subject: VPN disconnect issue - escalation needed

Hi team, this is Sarah Chen from Acme Logistics writing in again about
ticket TKT-1042. Our Gold-tier SLA promises a 4 hour response time, and
we are now at hour 6 with no update. The VPN client keeps dropping every
10 minutes on our Windows fleet since the rollout of CloudXeus Connect
v3.2 last Tuesday. If this isn't resolved by end of day Friday we will
be requesting the $500 SLA breach credit outlined in our contract.
"""

system_prompt = """
You are a text analysis engine for CloudXeus support tickets.
Read the ticket text and respond with ONLY a JSON object containing:
- "entities": a list of objects, each with "text" and "category"
  (categories: person, organization, date, product, ticket_id, sla_tier, monetary_amount)
- "topics": a list of short topic labels describing what the ticket is about

Respond with JSON only. Do not include any other text.
"""

### Step 3 — Call the Responses API and print the result

`openai.responses.create(...)` is the Azure OpenAI **Responses API** — a newer, unified API surface (alongside
the older Chat Completions API) that takes a list of typed input items (here, two `message` items: `system`
and `user`) and returns a response object. `response.output_text` is a convenience property that concatenates
the model's text output.

Note there is no schema enforcement here — the model is *asked* to return JSON-only, but nothing guarantees
it. In production you'd want to either parse defensively or use structured outputs (notebook 02).

💡 **Exam tip:** The Responses API is Azure OpenAI/OpenAI-specific — it has no equivalent in Azure AI Language.
For the exam, keep straight which SDK/API surface belongs to which service: Azure OpenAI (`openai` SDK,
`responses.create`/`chat.completions.create`) vs. Azure AI Language (`azure-ai-textanalytics`,
`recognize_entities`/`recognize_pii_entities`/etc.).

🔄 **Alternatives:** `openai.chat.completions.create(...)` — the older, still widely-used Chat Completions API
— would work equivalently for this simple case; Responses API adds features like built-in tool use and
stateful conversations that aren't exercised here.


In [ ]:
response = openai.responses.create(
    model=deployment_name,
    input=[
        {"type": "message", "role": "system", "content": system_prompt},
        {"type": "message", "role": "user", "content": ticket_text}
    ]
)

print(response.output_text)

## Summary

This notebook sent a support ticket to a chat-completion-class LLM via the Azure OpenAI Responses API and
asked it, purely through prompt instructions, to extract entities and topics as JSON text. The output is
whatever the model decided to produce — likely valid JSON matching the requested shape, but not guaranteed.
This is the "quick and flexible" end of the extraction spectrum: zero training data, arbitrary categories,
but no schema guarantee and no confidence scores per entity (unlike the dedicated Language service in
notebook 07).


## Try It Yourself

1. **Easy:** Change the `ticket_text` to a different support scenario (e.g. a billing complaint) and rerun —
   see how the model adapts its entity/topic choices.
2. **Intermediate:** Add a `try/except json.loads(response.output_text)` around the output to detect when the
   model fails to return valid JSON, and print a warning instead of crashing.
3. **Advanced:** Run both this notebook and `07_azure_language_ner.py` on the *same* ticket text, and write a
   short comparison of which entities each approach found, missed, or mis-categorized.
